# ZiCore Unsloth Fine-Tuning Prototype
Fine-tune Llama-3.1-8B or Mistral-7B on 1190 ZiCore training examples
Hardware: T4 GPU (free Colab)

In [ ]:
!pip install unsloth -q
!pip install xformers trl peft accelerate bitsandbytes -q

In [ ]:
import torch
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import json

max_seq_length = 512
dtype = None
load_in_4bit = True

## 1. Load Dataset
1190 examples across 17 ZiCore modules

In [ ]:
# Upload zicore_training.jsonl to Colab or mount Drive
from google.colab import drive
drive.mount('/content/drive')

data = []
with open('/content/drive/MyDrive/zicore_training.jsonl', 'r') as f:
    for line in f:
        data.append(json.loads(line))
print(f'Loaded {len(data)} examples')
modules = {}
for item in data:
    m = item['module']
    modules[m] = modules.get(m, 0) + 1
for m, c in sorted(modules.items()):
    print(f'  {m}: {c}')

In [ ]:
# Format for training
def format_example(example):
    return {
        'text': f"""### Module: {example['module']}
### Instruction: {example['instruction']}
### Response: {example['output']}"""
    }

dataset = Dataset.from_list(data).map(format_example)
dataset = dataset.train_test_split(test_size=0.1)
print(f'Train: {len(dataset["train"])}, Test: {len(dataset["test"])}')

## 2. Load Base Model with Unsloth

In [ ]:
# Choose one:
MODEL_NAME = "unsloth/Meta-Llama-3.1-8B-bnb-4bit"  # 8B
# MODEL_NAME = "unsloth/mistral-7b-v0.3-bnb-4bit"  # 7B

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)
print(f'Loaded {MODEL_NAME}')

In [ ]:
# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)
print(f'LoRA adapters added. Trainable params: {model.num_parameters(only_trainable=True):,}')

## 3. Train

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    dataset_text_field='text',
    max_seq_length=max_seq_length,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        eval_steps=50,
        save_steps=100,
        output_dir='zicore-lora',
        report_to='none',
    ),
)
trainer.train()

## 4. Evaluate

In [ ]:
import random

FastLanguageModel.for_inference(model)
test_samples = random.sample(dataset['test'], 5)
for s in test_samples:
    prompt = s['text'].split('### Response:')[0] + '### Response:'
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    outputs = model.generate(**inputs, max_new_tokens=128, temperature=0.7)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print('='*40)
    print(f'Expected: {s["output"][:100]}')
    print(f'Got: {response[len(prompt):][:100]}')

## 5. Save & Push to HuggingFace Hub

In [ ]:
# Save locally
model.save_pretrained('zicore-lora-final')
tokenizer.save_pretrained('zicore-lora-final')

# Merge with base model for standalone inference
from unsloth import FastLanguageModel
merged_model = FastLanguageModel.for_inference(model)
merged_model = merged_model.merge_and_unload()
merged_model.save_pretrained('zicore-merged')
tokenizer.save_pretrained('zicore-merged')
print('Model saved to zicore-merged/')

In [ ]:
# Push to HuggingFace Hub (requires HF token)
from huggingface_hub import login, HfApi
login()  # paste your token

repo_id = 'your-username/zicore-llama-8b'
model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)
print(f'Pushed to https://huggingface.co/{repo_id}')

## 6. Using the Fine-tuned Model in ZiCore Backend
Set environment variable:
```
ZICORE_MODEL_PATH=your-username/zicore-llama-8b
```
Engine B will automatically load the model and use it for inference.